# 9.7 · 优化器 / Optimizers

> **课程定位 / Where this fits**
> 第 7 课，**Part 9 · 深度学习基础**。
> Lesson 7, **Part 9 · Deep Learning Foundations**.
>
> 有了损失，用什么算法去最小化它？这就是**优化器**。从最朴素的 **SGD**，到加"惯性"的 **Momentum**，到自适应学习率的 **RMSProp/Adagrad**，再到集大成的 **Adam/AdamW**——这是一条清晰的演进线。理解每个优化器**解决了前一个的什么问题**，是深度学习训练的核心内功。
> Given a loss, what algorithm minimizes it? That's the **optimizer**. From plain **SGD**, to **Momentum** (adds "inertia"), to adaptive-LR **RMSProp/Adagrad**, to the all-in-one **Adam/AdamW** — a clear evolution. Understanding **what problem each one fixes in its predecessor** is core training know-how.
>
> 💼 **实战/面试视角**："SGD vs Adam / Momentum 干什么 / Adam 怎么工作 / AdamW 和 Adam 区别" 高频。
> 💼 **Practical/interview angle:** "SGD vs Adam / what Momentum does / how Adam works / AdamW vs Adam" — common.

> 📐 **符号约定 / Notation**
> - $g_t$ —— 第 $t$ 步的梯度 / gradient at step $t$
> - $m_t$ —— 一阶矩(动量)/ first moment (momentum)
> - $v_t$ —— 二阶矩(梯度平方的滑动平均)/ second moment
> - $\eta$ —— 学习率 / learning rate

> 💡 **面试相关 / Interview-relevant**
> - "SGD vs Adam 的区别与取舍"（出镜率 ★★★★★）
> - "Momentum 解决什么问题"（出镜率 ★★★★★）
> - "Adam 怎么工作（一阶矩+二阶矩）"（★★★★★）
> - "为什么 Adam 要偏差校正"（★★★★）
> - "AdamW 和 Adam 的区别（解耦权重衰减）"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 SGD 及其问题（震荡、慢）。
   Understand SGD and its problems (oscillation, slowness).
2. 理解 **Momentum** 如何用"惯性"加速、减震荡。
   Understand how **Momentum** accelerates and damps oscillation via inertia.
3. 理解 **自适应学习率**（RMSProp/Adagrad）。
   Understand adaptive learning rates (RMSProp/Adagrad).
4. **从零**实现 **Adam**（动量 + 自适应 + 偏差校正）。
   Implement **Adam** from scratch (momentum + adaptive + bias correction).
5. 理解 **AdamW**（解耦权重衰减）并在 Digits 上对比。
   Understand **AdamW** (decoupled weight decay) and compare on Digits.

## 目录 / TOC
1. [先建直觉：优化的难点 ⭐](#1)
2. [SGD + Momentum（2D 轨迹）⭐](#2)
3. [自适应学习率：RMSProp ⭐](#3)
4. [Adam（从零）⭐](#4)
5. [AdamW + 实测对比 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：优化的难点 ⭐ / Intuition: Why Optimization Is Hard

梯度下降的基本式 $\theta \leftarrow \theta - \eta\,g$ 看着简单，但深度网络的损失曲面**又高维又崎岖**，朴素 SGD 有三个典型困难：
The basic update $\theta \leftarrow \theta - \eta\,g$ looks simple, but deep nets' loss surfaces are **high-dimensional and rugged**, giving plain SGD three typical difficulties:
- **峡谷震荡**：损失曲面像一条狭长山谷（某方向陡、某方向缓），SGD 会在陡的方向**来回横跳**，沿缓的方向**爬得很慢**。
  **Ravine oscillation:** when the surface is a long narrow valley (steep one way, shallow another), SGD **zig-zags** across the steep direction and crawls along the shallow one.
- **学习率难调**：所有参数共用一个 $\eta$。太大震荡/发散，太小慢；而不同参数其实需要不同步长。
  **One LR for all:** all parameters share one $\eta$. Too big oscillates/diverges, too small is slow; yet different parameters need different step sizes.
- **鞍点/平坦区**：梯度接近 0 的地方，SGD 几乎停滞。
  **Saddles/plateaus:** where gradients are near 0, SGD stalls.

后面的优化器就是逐个解决这些问题：**Momentum 治震荡，自适应学习率治"一刀切的 η"，Adam 把两者合一**。
The later optimizers fix these one by one: **Momentum cures oscillation, adaptive LR cures the one-size-fits-all η, and Adam unifies both**.


<a id="2"></a>
## 2. SGD + Momentum（2D 轨迹）⭐ / SGD + Momentum

**Momentum（动量）** 给更新加"惯性"：不只看当前梯度，还**累积过去梯度的方向**（像一个滚下山的球）：
**Momentum** adds "inertia": instead of only the current gradient, it **accumulates past gradients' direction** (like a ball rolling downhill):

$$m_t = \beta m_{t-1} + g_t, \qquad \theta \leftarrow \theta - \eta\, m_t \quad(\beta\approx0.9)$$

效果：在**一致的下坡方向**上速度越积越快（加速），在**来回震荡的方向**上正负梯度相互抵消（减震荡）。用一个狭长峡谷的 2D 损失可视化 SGD vs Momentum 的轨迹，差别一目了然。
Effect: along a **consistent downhill direction** it accelerates (speed builds up); along an **oscillating direction** positive/negative gradients cancel (damps oscillation). We visualize SGD vs Momentum trajectories on a narrow-ravine 2D loss — the difference is stark.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# 一个狭长峡谷损失: f=a·x² + b·y² (a≫b → 在 x 方向陡, y 方向缓) / a narrow ravine
a, b = 8.0, 0.5
def f(x, y): return a*x**2 + b*y**2
def grad(x, y): return np.array([2*a*x, 2*b*y])

def run(opt_step, lr, n=40, start=(-4.5, -4.5)):
    p = np.array(start, float); path = [p.copy()]; state = {}
    for _ in range(n):
        g = grad(*p); p = opt_step(p, g, lr, state); path.append(p.copy())
    return np.array(path)

def sgd_step(p, g, lr, s): return p - lr*g
def momentum_step(p, g, lr, s):
    s["m"] = 0.9*s.get("m", 0) + g                       # 累积过去梯度方向(惯性)
    return p - lr*s["m"]

path_sgd = run(sgd_step, lr=0.06)
path_mom = run(momentum_step, lr=0.06)

xx, yy = np.meshgrid(np.linspace(-5,5,100), np.linspace(-5,5,100))
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.contour(xx, yy, f(xx, yy), levels=30, cmap="Blues", alpha=0.5)
ax.plot(path_sgd[:,0], path_sgd[:,1], "o-", color="C3", ms=3, label=f"SGD (震荡, {len(path_sgd)}步未到)")
ax.plot(path_mom[:,0], path_mom[:,1], "o-", color="C2", ms=3, label="Momentum (减震荡+加速)")
ax.scatter([0],[0], c="k", marker="*", s=200, label="最优点")
ax.legend(); ax.set_title("狭长峡谷: SGD 在陡方向来回横跳; Momentum 用惯性抵消震荡+加速")
plt.tight_layout(); plt.show()
print(f"40 步后到最优点的距离: SGD={np.linalg.norm(path_sgd[-1]):.3f}, Momentum={np.linalg.norm(path_mom[-1]):.3f}")
print("Momentum 累积梯度: 一致方向加速, 震荡方向(正负梯度)抵消 → 又快又稳")


<a id="3"></a>
## 3. 自适应学习率：RMSProp ⭐ / Adaptive LR: RMSProp

Momentum 解决了震荡，但所有参数还是**共用一个学习率**。**自适应学习率**的想法：**给每个参数单独调步长**——梯度一直很大的参数用小步长，梯度小的用大步长。
Momentum cured oscillation, but all parameters still **share one learning rate**. The **adaptive LR** idea: **a per-parameter step size** — parameters with consistently large gradients get small steps, those with small gradients get large steps.

**RMSProp** 维护每个参数**梯度平方的滑动平均** $v_t$，用它**归一化**步长：
**RMSProp** keeps a running average of each parameter's **squared gradient** $v_t$ and **normalizes** the step by it:

$$v_t = \beta v_{t-1} + (1-\beta)g_t^2, \qquad \theta \leftarrow \theta - \frac{\eta}{\sqrt{v_t}+\epsilon}\, g_t$$

梯度大的方向 $v_t$ 大、步长被压小（不再横跳）；梯度小的方向步长相对放大（不再龟速）。**Adagrad** 是它的前身（累积所有历史梯度平方，但会让学习率单调衰减到几乎为 0，RMSProp 用滑动平均修复了这点）。
A large-gradient direction has large $v_t$ and a shrunk step (no more zig-zag); a small-gradient direction gets a relatively larger step. **Adagrad** is the predecessor (accumulates all historical squared gradients, but its LR decays monotonically to ~0; RMSProp's running average fixes this).


In [ ]:
def rmsprop_step(p, g, lr, s, beta=0.9, eps=1e-8):
    s["v"] = beta*s.get("v", 0) + (1-beta)*g**2          # 梯度平方的滑动平均(每参数)
    return p - lr * g / (np.sqrt(s["v"]) + eps)          # 用 √v 归一化步长 → 每参数自适应

path_rms = run(rmsprop_step, lr=0.3)
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.contour(xx, yy, f(xx, yy), levels=30, cmap="Blues", alpha=0.5)
ax.plot(path_sgd[:,0], path_sgd[:,1], "o-", color="C3", ms=3, label="SGD")
ax.plot(path_rms[:,0], path_rms[:,1], "o-", color="C0", ms=3, label="RMSProp(每参数自适应步长)")
ax.scatter([0],[0], c="k", marker="*", s=200)
ax.legend(); ax.set_title("RMSProp: 用梯度平方归一化, 陡方向自动减小步长, 缓方向放大 → 各向均衡")
plt.tight_layout(); plt.show()
print(f"40 步后到最优点距离: RMSProp={np.linalg.norm(path_rms[-1]):.3f}")
print("RMSProp 给每个参数自适应步长(梯度大→步小, 梯度小→步大); Adagrad 是前身(但学习率会衰减到0)")


<a id="4"></a>
## 4. Adam（从零）⭐ / Adam from Scratch

**Adam = Momentum + RMSProp**——同时用**一阶矩**（动量，方向）和**二阶矩**（自适应步长），是目前最常用的优化器。它维护两个滑动平均：
**Adam = Momentum + RMSProp** — it uses both the **first moment** (momentum, direction) and the **second moment** (adaptive step), the most popular optimizer today. It keeps two running averages:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad(\text{动量}), \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2 \quad(\text{自适应})$$

**偏差校正(bias correction)**（面试要点）：$m_t, v_t$ 初始化为 0，前几步会**偏向 0**（被初始值拖累）。Adam 除以 $(1-\beta^t)$ 校正：$\hat m_t = m_t/(1-\beta_1^t)$，$\hat v_t = v_t/(1-\beta_2^t)$，让早期估计无偏。最终更新：
**Bias correction** (interview point): $m_t, v_t$ start at 0 and are **biased toward 0** in early steps. Adam divides by $(1-\beta^t)$: $\hat m_t = m_t/(1-\beta_1^t)$, $\hat v_t = v_t/(1-\beta_2^t)$, debiasing early estimates. The update:

$$\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat v_t}+\epsilon}\,\hat m_t$$


In [ ]:
def adam_step(p, g, lr, s, b1=0.9, b2=0.999, eps=1e-8):
    s["t"] = s.get("t", 0) + 1                           # 步数(偏差校正要用)
    s["m"] = b1*s.get("m", 0) + (1-b1)*g                 # 一阶矩(动量)
    s["v"] = b2*s.get("v", 0) + (1-b2)*g**2              # 二阶矩(自适应)
    m_hat = s["m"] / (1 - b1**s["t"])                    # 偏差校正(修正初期偏向0)
    v_hat = s["v"] / (1 - b2**s["t"])
    return p - lr * m_hat / (np.sqrt(v_hat) + eps)       # 动量方向 × 自适应步长

path_adam = run(adam_step, lr=0.5)
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.contour(xx, yy, f(xx, yy), levels=30, cmap="Blues", alpha=0.5)
for path, name, c in [(path_sgd,"SGD","C3"), (path_mom,"Momentum","C2"),
                      (path_rms,"RMSProp","C0"), (path_adam,"Adam","C1")]:
    ax.plot(path[:,0], path[:,1], "o-", ms=2.5, label=f"{name} (终距 {np.linalg.norm(path[-1]):.2f})")
ax.scatter([0],[0], c="k", marker="*", s=200); ax.legend(fontsize=8)
ax.set_title("四优化器轨迹对比: Adam(动量+自适应+偏差校正) 通常又快又稳")
plt.tight_layout(); plt.show()
print("Adam = Momentum(方向惯性) + RMSProp(每参数自适应步长) + 偏差校正(修初期)")
print("默认超参 β1=0.9, β2=0.999, ε=1e-8; 是深度学习最常用的优化器(快+鲁棒+少调参)")


<a id="5"></a>
## 5. AdamW + 实测对比 + 小结 ⭐ / AdamW & Comparison

**AdamW** 是 Adam 的重要改进（现代 Transformer 的标配，面试常问）。区别在**权重衰减(weight decay = L2 正则)** 的处理：
**AdamW** is an important Adam improvement (standard in modern Transformers, often asked). The difference is how **weight decay (= L2 regularization)** is applied:
- **Adam + L2**：把 L2 惩罚加进梯度里，再过自适应缩放——结果**权重衰减被自适应步长扭曲**（梯度大的参数被衰减得少），正则效果不纯。
  **Adam + L2:** adds the L2 penalty into the gradient, then through adaptive scaling — so **weight decay gets distorted by the adaptive step** (large-gradient params decay less), impure regularization.
- **AdamW**：把权重衰减**从梯度里解耦**，直接作用在权重上（$\theta \leftarrow \theta - \eta\lambda\theta$ 单独一项）。正则更干净、泛化更好。

  **AdamW:** **decouples** weight decay from the gradient, applying it directly to the weights (a separate $\theta \leftarrow \theta - \eta\lambda\theta$ term). Cleaner regularization, better generalization.

在 Digits 上对比各优化器的收敛。**实战建议**：**Adam/AdamW 是默认首选**（快、鲁棒、几乎不用调）；调好的 SGD+Momentum 有时泛化更好（CV 任务常见），但更难调。
On Digits we compare optimizers' convergence. **Practical advice:** **Adam/AdamW are the default** (fast, robust, near-tuning-free); a well-tuned SGD+Momentum can generalize better (common in CV) but is harder to tune.


In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits(); X = digits.data/16.0
X_tr, X_te, y_tr, y_te = train_test_split(X, digits.target, test_size=0.3, stratify=digits.target, random_state=0)
Xtr_t = torch.tensor(X_tr, dtype=torch.float32); ytr_t = torch.tensor(y_tr)
Xte_t = torch.tensor(X_te, dtype=torch.float32)

def make_net():
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,10))

ce = nn.CrossEntropyLoss()
optimizers = {
    "SGD":            lambda p: torch.optim.SGD(p, lr=0.1),
    "SGD+Momentum":   lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9),
    "RMSprop":        lambda p: torch.optim.RMSprop(p, lr=1e-3),
    "Adam":           lambda p: torch.optim.Adam(p, lr=1e-3),
    "AdamW":          lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=1e-2),
}
fig, ax = plt.subplots(figsize=(7.5, 4))
for name, make_opt in optimizers.items():
    net = make_net(); opt = make_opt(net.parameters()); losses = []
    for _ in range(60):
        opt.zero_grad(); loss = ce(net(Xtr_t), ytr_t); loss.backward(); opt.step()
        losses.append(loss.item())
    acc = (net(Xte_t).argmax(1).numpy() == y_te).mean()
    ax.plot(losses, label=f"{name} (test {acc:.3f})")
ax.set_xlabel("epoch"); ax.set_ylabel("训练损失"); ax.legend(fontsize=8); ax.set_yscale("log")
ax.set_title("各优化器收敛对比: Adam/AdamW 通常最快")
plt.tight_layout(); plt.show()
print("Adam/AdamW 通常收敛最快; AdamW 解耦权重衰减 → 正则更干净(Transformer 标配)")
print("实战: 默认 Adam/AdamW(快+鲁棒); 调好的 SGD+Momentum 有时泛化更好但难调")


```
优化难点: 峡谷震荡 / 一刀切学习率 / 鞍点平坦区 → 各优化器逐个解决
SGD: θ-=η·g; 简单但峡谷震荡+慢
Momentum: 累积过去梯度(惯性) → 一致方向加速, 震荡方向抵消
自适应学习率: 每参数单独步长; RMSProp(梯度平方滑动平均归一化), Adagrad(前身, LR 会衰减到0)
Adam = Momentum + RMSProp + 偏差校正(修 m,v 初期偏向0); 最常用
AdamW: 权重衰减从梯度解耦, 直接作用权重 → 正则更干净(Transformer 标配)
实战: 默认 Adam/AdamW; 调好的 SGD+Momentum 有时泛化更好(CV)但难调
```

### 💡 面试速查 / Interview cheat-sheet
1. **Momentum 累积梯度(惯性)** → 加速 + 减震荡。
   Momentum accumulates gradients (inertia) → accelerates + damps oscillation.
2. **自适应学习率(RMSProp/Adagrad)**: 每参数单独步长(梯度平方归一化)。
   Adaptive LR (RMSProp/Adagrad): per-parameter steps via squared-gradient normalization.
3. **Adam = 动量 + 自适应 + 偏差校正**(修 m,v 初期偏0)。
   Adam = momentum + adaptive + bias correction (fixes early bias toward 0).
4. **AdamW 解耦权重衰减** → 正则更干净, Transformer 标配。
   AdamW decouples weight decay → cleaner regularization, standard in Transformers.
5. **默认 Adam/AdamW**; 调好的 SGD+Momentum 有时泛化更好。
   Default to Adam/AdamW; well-tuned SGD+Momentum can generalize better.

### 下一节 / Next
**9.8 学习率调度**——优化器的学习率不该一成不变。warmup、cosine、step、one-cycle 等调度策略让训练更快更稳。
**9.8 LR Schedulers** — the learning rate shouldn't stay fixed. Warmup, cosine, step, one-cycle schedules make training faster and more stable.
